# Non-Linear Regression — Practice Set 3 🎯
## eBay iPhone Auctions Dataset

### Rules
- `alpha = 0.05`, `random_state = 617`, 75/25 split
- Use `train` unless stated otherwise
- Do NOT round; no commas; lead with 0

### About the Data
`eBayClean.csv` — eBay auction data for iPhones.
- `sold`: 1 if the item sold, 0 otherwise
- `startprice`: Starting auction price
- `charCountDescription`: Number of characters in description
- `upperCaseDescription`: Number of uppercase characters
- `condition`, `carrier`, `color`, `storage`, `productline` (categorical)

> **Goal**: Investigate how `startprice` non-linearly affects `charCountDescription` and also model `sold` using polynomial logistic regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.formula.api import ols
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

data = pd.read_csv('eBayClean.csv')
train = data.sample(frac=0.75, random_state=617)
test  = data.drop(labels=train.index)
print('Train:', train.shape, '| Test:', test.shape)
print(data.dtypes)

---
## ❓ Question 1
> Split data 75% train / 25% test using `data.sample()` with `random_state=617`.
>
> **What is the median `startprice` in the train sample?**

In [ ]:
print('Median startprice in train:', train.startprice.median())

---
## ❓ Question 2
> Construct a scatter plot with `startprice` on the x-axis and `charCountDescription` on the y-axis.
> **What shape does the relationship appear to take?**
>
> Also compute the Pearson correlation between the two. **Is it statistically significant?**

In [ ]:
from scipy.stats import pearsonr
sns.scatterplot(data=train, x='startprice', y='charCountDescription', s=8, alpha=0.4)
plt.show()

r, p = pearsonr(train.startprice, train.charCountDescription)
print(f'Pearson r={r:.4f}, p={p:.4e}')

---
## ❓ Question 3
> Fit two models predicting `charCountDescription` from `startprice`:
> - `model_lin`: Linear
> - `model_p2`: Degree-2 polynomial
>
> **What is the R² of each model?** Which fits better?

In [ ]:
model_lin = ols('charCountDescription ~ startprice', data=train).fit()
model_p2  = ols('charCountDescription ~ startprice + I(startprice**2)', data=train).fit()

print('Linear R²: ', model_lin.rsquared)
print('Poly-2 R²: ', model_p2.rsquared)

---
## ❓ Question 4
> Using `model_p2`, **what is the predicted `charCountDescription` for an item with `startprice = 200`?**

In [ ]:
new = pd.DataFrame({'startprice': [200]})
print('Predicted charCountDescription for startprice=200:', model_p2.predict(new).values[0])

---
## ❓ Question 5
> Create a **step function** by binning `startprice` into:
> `[0, 100)`, `[100, 300)`, `[300, 600)`, `[600+)` — labels: `low`, `mid`, `high`, `premium`.
>
> Fit `charCountDescription ~ price_bin`. Call this `model_step`.
>
> **Which price bin is significantly different from the reference (`low`) at α=0.05?**

In [ ]:
train = train.copy(); test = test.copy()
bins = [0, 100, 300, 600, float('inf')]
labels = ['low','mid','high','premium']
train['price_bin'] = pd.cut(train.startprice, bins=bins, labels=labels)
test['price_bin']  = pd.cut(test.startprice,  bins=bins, labels=labels)

model_step = ols('charCountDescription ~ price_bin', data=train).fit()
print(model_step.summary())

---
## ❓ Question 6
> Compare the test RMSE of `model_lin`, `model_p2`, and `model_step`.
>
> **Which model performs best on the test set?**

In [ ]:
def rmse(y, yh): return np.sqrt(np.mean((y-yh)**2))

df_res = pd.DataFrame({
    'model':['linear','poly2','step'],
    'train_rmse':[
        rmse(train.charCountDescription, model_lin.predict()),
        rmse(train.charCountDescription, model_p2.predict()),
        rmse(train.charCountDescription, model_step.predict()),
    ],
    'test_rmse':[
        rmse(test.charCountDescription, model_lin.predict(test)),
        rmse(test.charCountDescription, model_p2.predict(test)),
        rmse(test.charCountDescription, model_step.predict(test)),
    ]
}).set_index('model')
print(df_res)
print('Best test RMSE:', df_res.test_rmse.idxmin())

---
## ❓ Question 7
> Now use sklearn's `PolynomialFeatures(degree=3)` on `startprice`.
> Fit a `LinearRegression`. Call it `sk_poly3`.
>
> **What is the test RMSE of `sk_poly3`?**
> **Is degree-3 better than degree-2 on the test set?**

In [ ]:
# Degree 2
poly2 = PolynomialFeatures(degree=2, include_bias=False)
Xtr2 = poly2.fit_transform(train[['startprice']])
Xte2 = poly2.transform(test[['startprice']])
sk_poly2 = LinearRegression().fit(Xtr2, train.charCountDescription)

# Degree 3
poly3 = PolynomialFeatures(degree=3, include_bias=False)
Xtr3 = poly3.fit_transform(train[['startprice']])
Xte3 = poly3.transform(test[['startprice']])
sk_poly3 = LinearRegression().fit(Xtr3, train.charCountDescription)

print('sk_poly2 test RMSE:', root_mean_squared_error(test.charCountDescription, sk_poly2.predict(Xte2)))
print('sk_poly3 test RMSE:', root_mean_squared_error(test.charCountDescription, sk_poly3.predict(Xte3)))

---
## ❓ Question 8
> Use `PolynomialFeatures(degree=2)` on BOTH `startprice` and `upperCaseDescription`.
> Fit a `LinearRegression` predicting `charCountDescription`. Call it `sk_multi_poly2`.
>
> **How many columns are in the transformed feature matrix?**
> **What is the train RMSE?**

In [ ]:
poly_m = PolynomialFeatures(degree=2, include_bias=False)
Xtr_m = poly_m.fit_transform(train[['startprice','upperCaseDescription']])
Xte_m = poly_m.transform(test[['startprice','upperCaseDescription']])

print('Feature count:', Xtr_m.shape[1])
print('Feature names:', poly_m.get_feature_names_out())

sk_multi_poly2 = LinearRegression().fit(Xtr_m, train.charCountDescription)
print('Train RMSE:', root_mean_squared_error(train.charCountDescription, sk_multi_poly2.predict(Xtr_m)))
print('Test RMSE: ', root_mean_squared_error(test.charCountDescription,  sk_multi_poly2.predict(Xte_m)))

---
## ❓ Question 9
> Now consider the binary outcome `sold`. Using `ols()` with a **degree-2 polynomial** of `startprice`, construct a **linear probability model** (OLS) to predict `sold`.
>
> Note: This is NOT logistic regression — just OLS with a binary outcome.
>
> **Is the quadratic term significant? What does the coefficient sign tell you?**

In [ ]:
model_sold_poly2 = ols('sold ~ startprice + I(startprice**2)', data=train).fit()
print(model_sold_poly2.summary())

---
## ❓ Question 10
> Among all models you built in this assignment, summarize the **train vs test RMSE** for predicting `charCountDescription`.
>
> **Which combination of method + features gives the best test performance?**
>
> **Does adding more polynomial features always improve test RMSE?**

In [ ]:
final = pd.DataFrame({
    'model':['linear','poly2_sm','step','sk_poly2','sk_poly3','sk_multi_poly2'],
    'test_rmse':[
        rmse(test.charCountDescription, model_lin.predict(test)),
        rmse(test.charCountDescription, model_p2.predict(test)),
        rmse(test.charCountDescription, model_step.predict(test)),
        root_mean_squared_error(test.charCountDescription, sk_poly2.predict(Xte2)),
        root_mean_squared_error(test.charCountDescription, sk_poly3.predict(Xte3)),
        root_mean_squared_error(test.charCountDescription, sk_multi_poly2.predict(Xte_m)),
    ]
}).set_index('model').sort_values('test_rmse')
print(final)
# Answer: more features don't always help test RMSE (overfitting risk)

---
# ✅ Key Concepts

| Question | Core Concept |
|---|---|
| 1-2 | EDA before modeling |
| 3-4 | OLS polynomial via `I()` syntax |
| 5 | Step function = categorical variable via `pd.cut()` |
| 6 | Model comparison via test RMSE |
| 7 | sklearn poly, degree comparison |
| 8 | Multi-variable polynomial (5 features from 2 variables) |
| 9 | Polynomial terms can be applied to any regression |
| 10 | Overfitting danger with high-degree polynomials |

## ⚠️ Exam Traps
- 2 variables, degree 2 → 5 features: x1, x2, x1², x1x2, x2²
- 2 variables, degree 3 → 9 features
- `pd.cut()` with `float('inf')` as the upper bound for open-ended bins
- Step function coefficient = difference from REFERENCE group
- Higher degree doesn't always mean lower test RMSE